# 08 — Моя EDA полного журнала InfraPulse

    **Цель:** увидеть все доступные годы 2019–2026, отдельно прочитать готовый Parquet и исходный CSV,
    сверить их, затем изучить четыре дополнительных CSV и связи с годовым журналом.
    Включение 2021 здесь относится только к описательной EDA: текущая политика обучения исключает его.

    Основные `parquet_df` и `csv_df` ниже — **единые ленивые DuckDB relations**. Они ведут себя как
    табличные датафреймы для SQL/фильтров, но не загружают ~313 млн строк в RAM. `df()` вызывается
    только для маленьких итоговых таблиц. Никакой дедупликации: повторная запись остаётся записью.
    Исходные значения и флаг тревоги сохраняются раздельно. `alarm=true` не означает подтверждённый отказ.

    Запуск из корня: `uv sync --locked --group train --group research`, затем `make research`.
    Входы: `data-science/data/parquet/year=YYYY/*.parquet` и локальные, игнорируемые Git
    `data-science/data/raw/*.csv`. Ячейки не меняют исходные файлы и не обучают модель.
    Годовые CSV здесь уже распакованы; архивы `.7z` доступны в той же папке для проверки происхождения.

## Актуальный подготовленный журнал — открыть сначала

Этот блок показывает **текущую публикацию** `journal-curated-v4` с исследовательским overlay `alarm-spike-exclusions-v1`. Ниже по блокноту остаётся прежняя полная EDA исходных CSV и старого исследовательского Parquet. Её `enriched_df` — отдельный исторический обзор, **не** актуальный рабочий слой.

`current_enriched_df` — одно ленивое DuckDB-представление на уровне исходного события. В нём сохранены `alarm`, `value_raw`, nullable `value_numeric`, происхождение и quality flags подготовленного слоя; текущие справочники добавляют название канала и объекта. Совпадение со справочником состояний — диагностический кандидат, не исправленный `alarm` и не метка отказа. Журнал-пример и планы ППР/ТО не имеют подтверждённой связи с историей и сюда не присоединяются. Принадлежность канала объекту в прошлом по текущему снимку не доказана.

Локальные входы лежат в игнорируемом Git `data-science/artifacts/`. Последний день исходных событий — **30.06.2026**. Данных за июль–декабрь 2026 здесь нет. Полный `.df()` для сотен миллионов строк не вызывайте: схема и `LIMIT` ниже показывают состав без загрузки всей таблицы в RAM.

In [ ]:
import importlib
import json
import sys
from pathlib import Path

from IPython.display import display

CURRENT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "uv.lock").exists() and (p / "data-science").exists()
)
sys.path.insert(0, str(CURRENT_ROOT / "data-science/src"))
open_prepared = importlib.import_module("infra_pulse_research.data.prepared").open_prepared

CURRENT_SNAPSHOT = CURRENT_ROOT / "data-science/artifacts/curated-all-sensors-exclude-2021-q2"
CURRENT_POLICY = CURRENT_ROOT / "data-science/artifacts/alarm-spike-exclusions-v1/manifest.json"
STATE_CANDIDATES = CURRENT_ROOT / "data-science/artifacts/state-candidates-v1"
current_manifest = json.loads((CURRENT_SNAPSHOT / "manifest.json").read_text(encoding="utf-8"))
current_policy_manifest = json.loads(CURRENT_POLICY.read_text(encoding="utf-8"))
current_con = open_prepared(CURRENT_SNAPSHOT, policy_manifest=CURRENT_POLICY)
print("Curated:", CURRENT_SNAPSHOT.name, "| overlay:", CURRENT_POLICY.parent.name)
latest_month = max(
    p.name.removeprefix("month=") for p in (CURRENT_SNAPSHOT / "curated").glob("month=*")
)
last_source_timestamp = current_con.execute(
    "SELECT MAX(event_ts_local_raw) FROM working_events WHERE month = ?", [latest_month]
).fetchone()[0]
print("Последняя запись источника:", last_source_timestamp)
print("Рабочих записей по policy manifest:", current_policy_manifest["working_curated_rows"])
print("Текущие события — ленивое working_events; 2021 вне апреля–июня пока не допущен к обучению.")

In [ ]:
def _sql_path(path):
    return "'" + str(path).replace("'", "''") + "'"


current_con.execute(f"""
CREATE OR REPLACE VIEW current_channels AS
SELECT channel_id, name_raw AS sensor_name_current,
       tag_raw AS engineering_system_tag_current
FROM read_parquet({_sql_path(CURRENT_SNAPSHOT / "channels.parquet")})
""")
current_con.execute(f"""
CREATE OR REPLACE VIEW current_objects AS
SELECT TRY_CAST("ид_объект" AS BIGINT) AS object_id,
       TRY_CAST("иерархия_уровень" AS INTEGER) AS object_hierarchy_level,
       TRY_CAST("родитель" AS BIGINT) AS parent_object_id,
       "вид_объекта" AS object_kind,
       "диспетчерское_название_объекта" AS object_dispatch_name
FROM read_parquet({_sql_path(CURRENT_SNAPSHOT / "objects.parquet")})
""")
current_con.execute(f"""
CREATE OR REPLACE VIEW current_states AS
SELECT s.sensor_type, s.state_name, MIN(s.candidate_rows) AS candidate_rows,
       MIN(s.candidate_set_count) AS candidate_set_count,
       BOOL_AND(s.reference_alarm_consensus) AS reference_alarm_consensus,
       BOOL_OR(s.reference_alarm_conflict) AS reference_alarm_conflict,
       CASE WHEN MIN(s.candidate_set_count) = 1 THEN MIN(r.state_set_id) END AS candidate_set_id
FROM read_parquet({_sql_path(STATE_CANDIDATES / "exact_text_candidates.parquet")}) s
LEFT JOIN read_parquet({_sql_path(STATE_CANDIDATES / "source_rows.parquet")}) r
  ON s.sensor_type = r.sensor_type AND s.state_name = r.state_name
GROUP BY s.sensor_type, s.state_name
""")
for table, key in (("current_channels", "channel_id"), ("current_objects", "object_id")):
    total, distinct = current_con.execute(
        f"SELECT COUNT(*), COUNT(DISTINCT {key}) FROM {table}"
    ).fetchone()
    assert total == distinct, f"Неоднозначный ключ в {table}"
total, distinct = current_con.execute("""
    SELECT COUNT(*), COUNT(DISTINCT (sensor_type, state_name)) FROM current_states
""").fetchone()
assert total == distinct, "Неоднозначная пара тип × текст в справочнике состояний"

current_con.execute("""
CREATE OR REPLACE VIEW current_enriched_events AS
SELECT e.*,
       c.sensor_name_current, c.engineering_system_tag_current,
       o.object_hierarchy_level, o.parent_object_id,
       o.object_kind, o.object_dispatch_name,
       c.channel_id IS NOT NULL AS channel_ref_matched,
       o.object_id IS NOT NULL AS object_ref_matched,
       s.state_name IS NOT NULL AS state_ref_exact_match,
       s.candidate_rows AS state_ref_rows,
       s.candidate_set_count AS state_ref_set_count,
       s.candidate_set_id AS state_ref_candidate_set_id,
       s.reference_alarm_consensus AS state_ref_alarm_consensus,
       s.reference_alarm_conflict AS state_ref_alarm_conflict,
       'current_snapshot_no_validity_dates' AS reference_basis
FROM working_events e
LEFT JOIN current_channels c ON e.channel_id = c.channel_id
LEFT JOIN current_objects o ON e.object_id = o.object_id
LEFT JOIN current_states s
  ON e.sensor_type = s.sensor_type AND e.value_raw = s.state_name
""")
current_enriched_df = current_con.sql("SELECT * FROM current_enriched_events")
print(
    "current_enriched_df:",
    len(current_enriched_df.columns),
    "полей; соединения не размножают строки по ключам.",
)

In [ ]:
print("Схема актуального обогащённого слоя:")
display(current_con.sql("DESCRIBE current_enriched_events").df()[["column_name", "column_type"]])
print("Ограниченный просмотр 10 событий:")
display(
    current_con.sql("""
    SELECT month, event_ts_local_raw, channel_id, alarm, value_raw, value_numeric,
           system_type, sensor_type, sensor_name_current, object_id,
           object_kind, object_dispatch_name, channel_ref_matched,
           object_ref_matched, state_ref_exact_match,
           state_ref_candidate_set_id, state_ref_alarm_consensus,
           reference_basis
    FROM current_enriched_events LIMIT 10
""").df()
)
print("Для анализа: current_enriched_df.filter(...).limit(20).df()")

In [ ]:
import csv
from contextlib import contextmanager
from pathlib import Path
from threading import Event, Thread
from time import perf_counter

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    (
        p
        for p in (Path.cwd(), *Path.cwd().parents)
        if (p / "uv.lock").is_file() and (p / "data-science").is_dir()
    ),
    None,
)
if ROOT is None:
    raise RuntimeError("Откройте notebook внутри clone InfraPulse")
RAW = ROOT / "data-science/data/raw"
PARQUET = ROOT / "data-science/data/parquet"
YEARS = tuple(range(2019, 2027))
csv_files = {y: RAW / f"ext-journal-{y}.csv" for y in YEARS}
parquet_files = {y: sorted((PARQUET / f"year={y}").glob("*.parquet")) for y in YEARS}
required_refs = [
    "справочник_состояний.csv",
    "справочник_каналов_датчиков.csv",
    "журнал_событий_пример.csv",
    "справочник_объектов_диспетчер.csv",
]
missing = [str(p) for p in csv_files.values() if not p.is_file()]
missing += [f"year={y}/*.parquet" for y, paths in parquet_files.items() if not paths]
missing += [str(RAW / n) for n in required_refs if not (RAW / n).is_file()]
if missing:
    raise FileNotFoundError("Недоступные входы:\n" + "\n".join(missing))

print(f"Корень: {ROOT}")
print(
    f"Годовых CSV: {len(csv_files)}; "
    f"годовых Parquet: {sum(map(len, parquet_files.values()))}; "
    f"дополнительных CSV: {len(required_refs)}"
)
print("Размеры CSV, ГБ:", {y: round(p.stat().st_size / 1e9, 2) for y, p in csv_files.items()})
print("DuckDB", duckdb.__version__)

In [ ]:
# DuckDB сканирует колонки и файлы по частям; временные файлы остаются вне Git.
TEMP = ROOT / "data-science/artifacts/my-eda/tmp"
TEMP.mkdir(parents=True, exist_ok=True)
con = duckdb.connect()
con.execute("SET threads = 4")
con.execute("SET memory_limit = '4GB'")
con.execute("SET temp_directory = ?", [str(TEMP)])


# Индикатор DuckDB внутри Jupyter требует ipywidgets. Простой таймер работает
# с проектной .venv без дополнительных пакетов и печатает прогресс долгих SQL.
@contextmanager
def progress(label, interval_seconds=2):
    stop = Event()
    started = perf_counter()

    def report_elapsed():
        while not stop.wait(interval_seconds):
            print(f"{label}: выполняется {perf_counter() - started:.0f} с...", flush=True)

    reporter = Thread(target=report_elapsed, daemon=True)
    reporter.start()
    try:
        yield
    finally:
        stop.set()
        reporter.join(timeout=1)


def fetch_small(sql, label):
    """Выполнить долгий SQL и вернуть только небольшой агрегат."""
    with progress(label):
        return con.sql(sql).df()


def sql_string(value):
    return "'" + str(value).replace("'", "''") + "'"


def compact(sql, limit=30):
    """Преобразовывать в pandas только маленький агрегат, никогда полную историю."""
    return con.sql(sql + f" LIMIT {limit}").df()

## 1. Parquet: один ленивый датафрейм всех годов

    Берём только `year=YYYY/*.parquet`: соседние team snapshots и curated artifacts имеют другую
    область данных. Шесть общих полей нормализуем для сравнения с CSV; дополнительные колонки
    Parquet (`system_type`, `sensor_type`, `value_numeric` и др.) остаются доступны в исходных файлах.
    Время сохраняется как локальный исходный текст; подтверждённой временной зоны в файлах нет.

In [ ]:
def parquet_source(year):
    paths = ", ".join(sql_string(p) for p in parquet_files[year])
    return f"""
      SELECT {year}::INTEGER AS source_year,
             CAST(event_id AS VARCHAR) AS event_id,
             CAST(channel_id AS VARCHAR) AS channel_id,
             CAST(event_date AS VARCHAR) AS event_date,
             CAST(event_time AS VARCHAR) AS event_time,
             CASE WHEN is_alarm THEN 'true' WHEN NOT is_alarm THEN 'false' END AS alarm,
             raw_value AS value_raw
      FROM read_parquet([{paths}], union_by_name=true)
    """


con.execute(
    "CREATE OR REPLACE VIEW parquet_all AS " + " UNION ALL ".join(parquet_source(y) for y in YEARS)
)
parquet_df = con.sql("SELECT * FROM parquet_all")
print("parquet_df — единый ленивый датафрейм; схема:")
display(parquet_df.limit(0).df().dtypes)
display(parquet_df.limit(5).df())

In [ ]:
PROFILE = """
SELECT source_year,
       COUNT(*) AS rows,
       COUNT(DISTINCT channel_id) AS channels,
       MIN(event_date || ' ' || event_time) AS first_local_time,
       MAX(event_date || ' ' || event_time) AS last_local_time,
       COUNT_IF(alarm = 'true') AS alarm_true,
       COUNT_IF(alarm NOT IN ('true', 'false') OR alarm IS NULL) AS alarm_unknown,
       COUNT_IF(event_id IS NULL OR TRY_CAST(event_id AS BIGINT) IS NULL) AS bad_event_id,
       COUNT_IF(channel_id IS NULL OR TRY_CAST(channel_id AS BIGINT) IS NULL) AS bad_channel_id,
       COUNT_IF(value_raw IS NULL) AS null_value,
       COUNT_IF(TRY_CAST(value_raw AS DOUBLE) IS NOT NULL) AS numeric_value,
       SUM(hash(event_id, channel_id, event_date, event_time, alarm,
                value_raw)::HUGEINT)::VARCHAR AS hash_sum,
       BIT_XOR(hash(event_id, channel_id, event_date, event_time, alarm,
                    value_raw))::VARCHAR AS hash_xor
FROM {source}
GROUP BY source_year
"""


def profile_year(source, year, label):
    started = perf_counter()
    print(f"{label} {year}: сканирование...", flush=True)
    with progress(f"{label} {year}"):
        row = con.sql(PROFILE.format(source=source)).fetchone()
    names = [item[0] for item in con.sql(PROFILE.format(source=source) + " LIMIT 0").description]
    result = dict(zip(names, row, strict=True))
    result["seconds"] = round(perf_counter() - started, 1)
    print(f"{label} {year}: {result['rows']:,} записей, {result['seconds']} с", flush=True)
    return result


parquet_stats = [profile_year(f"({parquet_source(y)})", y, "Parquet") for y in YEARS]
display(
    duckdb.sql("SELECT * FROM parquet_stats").df()
    if False
    else pd.DataFrame(
        [{k: v for k, v in r.items() if not k.startswith("hash_")} for r in parquet_stats]
    )
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.bar(
    [r["source_year"] for r in parquet_stats],
    [r["rows"] / 1e6 for r in parquet_stats],
    color="#176a8a",
)
ax.set(title="История Parquet: записи по году", xlabel="Год источника", ylabel="Млн записей")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()
print(
    f"Всего Parquet: {sum(r['rows'] for r in parquet_stats):,} записей; "
    f"диапазон: {min(r['first_local_time'] for r in parquet_stats)} — "
    f"{max(r['last_local_time'] for r in parquet_stats)}"
)

## 2. Исходные CSV: самостоятельное чтение всех восьми лет

    DuckDB читает шесть полей как строки без вывода типов по выборке: ID не проходят через float,
    `значение_датчика` не теряет лексемы. Повтор технического заголовка внутри файла считаем
    отдельно и исключаем только из таблицы записей. `t/f` и `true/false` приводим к одному
    булевому тексту для сравнения. Остальные значения флага остаются `NULL` и видны в статистике.

In [ ]:
HEADER = ["ид_события", "ид_канала_данных", "дата", "время", "тревожное", "значение_датчика"]
for year, path in csv_files.items():
    with path.open(encoding="utf-8-sig", newline="") as stream:
        actual = next(csv.reader(stream))
    if actual != HEADER:
        raise ValueError(f"Неверный заголовок {year}: {actual!r}")

TECHNICAL_ROW = " AND ".join(f'"{name}" = {sql_string(name)}' for name in HEADER)
CSV_COLUMNS = "{" + ", ".join(f"{sql_string(c)}: 'VARCHAR'" for c in HEADER) + "}"


def csv_read(year):
    return (
        f"read_csv({sql_string(csv_files[year])}, header=true, auto_detect=false, "
        f"columns={CSV_COLUMNS}, delim=',', quote='\"', escape='\"', filename=true)"
    )


def csv_source(year):
    return f"""
      SELECT {year}::INTEGER AS source_year,
             "ид_события" AS event_id,
             "ид_канала_данных" AS channel_id,
             "дата" AS event_date,
             "время" AS event_time,
             CASE lower(trim("тревожное"))
                WHEN 't' THEN 'true' WHEN 'true' THEN 'true'
                WHEN 'f' THEN 'false' WHEN 'false' THEN 'false'
             END AS alarm,
             "значение_датчика" AS value_raw
      FROM {csv_read(year)}
      WHERE NOT ({TECHNICAL_ROW})
    """


con.execute("CREATE OR REPLACE VIEW csv_all AS " + " UNION ALL ".join(csv_source(y) for y in YEARS))
csv_df = con.sql("SELECT * FROM csv_all")
print("csv_df — единый ленивый датафрейм, как parquet_df:")
display(csv_df.limit(5).df())

In [ ]:
csv_stats = []
for year in YEARS:
    stats = profile_year(f"({csv_source(year)})", year, "CSV")
    with progress(f"CSV {year}: поиск повторного заголовка"):
        stats["technical_headers"] = con.sql(
            f"""SELECT COUNT(*) FROM {csv_read(year)} WHERE {TECHNICAL_ROW}"""
        ).fetchone()[0]
    csv_stats.append(stats)
display(
    pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("hash_")} for r in csv_stats])
)
print(
    f"Всего CSV-записей: {sum(r['rows'] for r in csv_stats):,}; "
    f"повторных технических заголовков: {sum(r['technical_headers'] for r in csv_stats)}"
)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(
    YEARS, [r["alarm_true"] / r["rows"] * 100 for r in csv_stats], marker="o", lw=2, color="#b04a25"
)
ax.set(
    title="Исходный флаг тревоги в годовых CSV",
    xlabel="Год источника",
    ylabel="Доля записей с alarm=true, %",
)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 3. Сравнение CSV и Parquet

    Сравниваем число строк, каналов, диапазон локального времени, тревоги, пропуски,
    распознаваемые числа и два независимых коммутативных отпечатка **всех шести общих полей**.
    Отпечатки учитывают повторы и порядок не важен, но это не криптографическое доказательство
    полного равенства; при расхождении нужен разбор строк и происхождения конвертера.
    Для строгой проверки предусмотрен выборочный/годовой multiset-запрос ниже.

In [ ]:
compare_columns = [
    "rows",
    "channels",
    "first_local_time",
    "last_local_time",
    "alarm_true",
    "alarm_unknown",
    "bad_event_id",
    "bad_channel_id",
    "null_value",
    "numeric_value",
    "hash_sum",
    "hash_xor",
]
comparison = []
for p, c in zip(parquet_stats, csv_stats, strict=True):
    row = {
        "year": p["source_year"],
        "csv_rows": c["rows"],
        "parquet_rows": p["rows"],
        "delta_rows": c["rows"] - p["rows"],
    }
    row.update({name + "_equal": c[name] == p[name] for name in compare_columns})
    row["all_checked_equal"] = all(row[name + "_equal"] for name in compare_columns)
    comparison.append(row)
display(pd.DataFrame(comparison))
for row in comparison:
    differences = [name for name in compare_columns if not row[name + "_equal"]]
    print(
        f"{row['year']}: "
        + (
            "проверенные агрегаты совпали"
            if not differences
            else "расхождения: " + ", ".join(differences)
        )
    )

In [ ]:
# Точный multiset audit одного выбранного года — затратен: полная сортировка/агрегация обеих сторон.
# Запускайте адресно после проверки таблицы выше, например exact_multiset_diff(2025).
def exact_multiset_diff(year):
    if year not in YEARS:
        raise ValueError(year)
    fields = "event_id, channel_id, event_date, event_time, alarm, value_raw"
    query = f"""
      WITH c AS (SELECT {fields}, COUNT(*) n FROM csv_all WHERE source_year={year} GROUP BY ALL),
           p AS (SELECT {fields}, COUNT(*) n FROM parquet_all
                    WHERE source_year={year} GROUP BY ALL),
           d AS ((SELECT * FROM c EXCEPT SELECT * FROM p)
                 UNION ALL (SELECT * FROM p EXCEPT SELECT * FROM c))
      SELECT COUNT(*) AS differing_distinct_rows FROM d
    """
    return con.sql(query).df()

## 4. Четыре дополнительных CSV

    Справочники — текущий снимок, историческая привязка каналов к объектам не доказана.
    Журнал-пример — отдельная выборка, не дополнительный девятый год и не обучающие исходы.
    Таблицы невелики, поэтому здесь можно показывать их агрегаты целиком.

In [ ]:
ref_paths = {
    "states": RAW / "справочник_состояний.csv",
    "channels": RAW / "справочник_каналов_датчиков.csv",
    "example": RAW / "журнал_событий_пример.csv",
    "objects": RAW / "справочник_объектов_диспетчер.csv",
}
for table, path in ref_paths.items():
    con.execute(
        f"CREATE OR REPLACE TABLE {table} AS "
        f"SELECT * FROM read_csv_auto({sql_string(path)}, all_varchar=true, header=true)"
    )
    print(
        table,
        con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0],
        "строк; поля:",
        con.sql(f"SELECT * FROM {table} LIMIT 0").columns,
    )

print("Повторы ключей и пустые связи:")
display(
    compact("""
SELECT
 (SELECT COUNT(*) - COUNT(DISTINCT "ид_канала_данных") FROM channels) AS duplicate_channel_ids,
 (SELECT COUNT(*) - COUNT(DISTINCT "ид_объект") FROM objects) AS duplicate_object_ids,
 (SELECT COUNT(*) FROM channels WHERE "ид_объект" IS NULL OR "ид_объект"='')
    AS channels_without_object_id,
 (SELECT COUNT(*) FROM channels c LEFT JOIN objects o ON c."ид_объект"=o."ид_объект"
    WHERE o."ид_объект" IS NULL) AS channels_unmatched_current_object,
 (SELECT COUNT(*) FROM (SELECT "тип_датчика", "название_состояния", COUNT(*) n
     FROM states GROUP BY ALL HAVING n>1)) AS repeated_state_type_text_pairs
""")
)

In [ ]:
print("Группы объектов — вид × уровень:")
object_groups = compact(
    """SELECT "вид_объекта" AS object_kind, "иерархия_уровень" AS level,
 COUNT(*) AS objects, COUNT(DISTINCT "родитель") AS parents
 FROM objects GROUP BY ALL ORDER BY objects DESC""",
    200,
)
display(object_groups)
print("Группы объектов по непосредственному родителю (топ-20):")
display(
    compact(
        """SELECT o."родитель" AS parent_id, p."вид_объекта" AS parent_kind,
 COUNT(*) AS child_objects, COUNT(DISTINCT o."вид_объекта") AS child_kinds
 FROM objects o LEFT JOIN objects p ON o."родитель"=p."ид_объект"
 GROUP BY ALL ORDER BY child_objects DESC""",
        20,
    )
)
print("Группы подсистем:")
system_groups = compact(
    """SELECT "тип_инж_системы" AS system_type,
 COUNT(*) AS channels, COUNT(DISTINCT "ид_объект") AS objects,
 COUNT(DISTINCT "тип_датчика") AS sensor_types
 FROM channels GROUP BY ALL ORDER BY channels DESC""",
    100,
)
display(system_groups)
print("Группы датчиков (топ-30; полный результат доступен через SQL):")
sensor_groups = compact(
    """SELECT "тип_инж_системы" AS system_type,
 "тип_датчика" AS sensor_type, COUNT(*) AS channels,
 COUNT(DISTINCT "ид_объект") AS objects
 FROM channels GROUP BY ALL ORDER BY channels DESC""",
    30,
)
display(sensor_groups)

In [ ]:
print("Состояния: покрытие по типам и alarm-метке справочника:")
display(
    compact(
        """SELECT "тип_датчика" AS sensor_type, "тревожное" AS state_alarm,
 COUNT(*) AS state_rows, COUNT(DISTINCT "название_состояния") AS texts
 FROM states GROUP BY ALL ORDER BY state_rows DESC""",
        100,
    )
)
print("Журнал-пример — отдельно от годовой истории:")
display(
    compact("""SELECT COUNT(*) AS rows, COUNT(DISTINCT "ид_канала_данных") AS channels,
 MIN("дата" || ' ' || "время") AS first_local_time,
 MAX("дата" || ' ' || "время") AS last_local_time,
 COUNT_IF(lower("тревожное") IN ('t','true')) AS alarm_true,
 COUNT_IF(TRY_CAST("значение_датчика" AS DOUBLE) IS NOT NULL) AS numeric_values
 FROM example""")
)
display(
    compact(
        """SELECT c."тип_инж_системы" AS system_type,
 c."тип_датчика" AS sensor_type, COUNT(*) AS rows,
 COUNT(DISTINCT e."ид_канала_данных") AS channels
 FROM example e LEFT JOIN channels c ON e."ид_канала_данных"=c."ид_канала_данных"
 GROUP BY ALL ORDER BY rows DESC""",
        25,
    )
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top_s = system_groups.head(12).iloc[::-1]
axes[0].barh(top_s["system_type"].fillna("Неизвестно"), top_s["channels"], color="#176a8a")
axes[0].set(title="Текущий справочник: каналы по подсистемам", xlabel="Число каналов")
top_t = (
    sensor_groups.groupby("sensor_type", as_index=False)["channels"]
    .sum()
    .nlargest(12, "channels")
    .iloc[::-1]
)
axes[1].barh(top_t["sensor_type"].fillna("Неизвестно"), top_t["channels"], color="#b04a25")
axes[1].set(title="Текущий справочник: каналы по типам датчиков", xlabel="Число каналов")
for ax in axes:
    ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 5. Первое сопоставление с годовым журналом

    Соединяем события по `channel_id` с текущим справочником каналов, затем по `ид_объект`
    с текущим справочником объектов. **Не присваиваем историческую принадлежность как факт:**
    смотрим покрытие и распределения с пометкой «по текущему справочнику».
    Если ключи справочников неуникальны, ниже прерываем join, чтобы не размножать строки.

In [ ]:
if con.sql('SELECT COUNT(*) - COUNT(DISTINCT "ид_канала_данных") FROM channels').fetchone()[0]:
    raise RuntimeError("Каналы имеют повторные ключи: join размножит строки")
if con.sql('SELECT COUNT(*) - COUNT(DISTINCT "ид_объект") FROM objects').fetchone()[0]:
    raise RuntimeError("Объекты имеют повторные ключи: join размножит строки")

started = perf_counter()
print("Сопоставление всей истории Parquet с текущими справочниками...", flush=True)
year_groups = fetch_small(
    """
SELECT p.source_year, COALESCE(c."тип_инж_системы", 'Нет в справочнике') AS system_type,
       COALESCE(c."тип_датчика", 'Нет в справочнике') AS sensor_type,
       COALESCE(o."вид_объекта", 'Нет объекта') AS object_kind,
       COUNT(*) AS rows, COUNT_IF(p.alarm='true') AS alarm_true,
       COUNT_IF(c."ид_канала_данных" IS NULL) AS unmatched_channel,
       COUNT_IF(c."ид_канала_данных" IS NOT NULL AND o."ид_объект" IS NULL) AS unmatched_object
FROM parquet_all p
LEFT JOIN channels c ON p.channel_id=c."ид_канала_данных"
LEFT JOIN objects o ON c."ид_объект"=o."ид_объект"
GROUP BY ALL
""",
    "Сопоставление истории со справочниками",
)  # малый агрегат
print(f"Готово за {perf_counter() - started:.1f} с; строк агрегата: {len(year_groups)}")
display(
    year_groups.groupby("source_year", as_index=False)[
        ["rows", "alarm_true", "unmatched_channel", "unmatched_object"]
    ].sum()
)
display(
    year_groups.groupby("system_type", as_index=False)[["rows", "alarm_true"]]
    .sum()
    .nlargest(15, "rows")
)
print("По видам объектов текущего справочника:")
display(
    year_groups.groupby("object_kind", as_index=False)[["rows", "alarm_true"]]
    .sum()
    .nlargest(15, "rows")
)
print("По типам датчиков текущего справочника (топ-20):")
display(
    year_groups.groupby("sensor_type", as_index=False)[["rows", "alarm_true"]]
    .sum()
    .nlargest(20, "rows")
)

In [ ]:
# Потенциальное совпадение со справочником состояний, без трактовки его как полной разметки.
# Дедупликация пары тип × текст предотвращает размножение событий при повторных строках справочника.
print("Проверка точных текстовых совпадений с текущим справочником состояний...", flush=True)
state_coverage = fetch_small(
    """
WITH state_pairs AS (
  SELECT DISTINCT "тип_датчика" AS sensor_type, "название_состояния" AS value_raw
  FROM states
), by_type AS (
  SELECT p.source_year, COALESCE(c."тип_датчика", 'Нет в справочнике') AS sensor_type,
         COUNT(*) AS rows,
         COUNT_IF(s.sensor_type IS NOT NULL) AS exact_state_text_matches
  FROM parquet_all p
  LEFT JOIN channels c ON p.channel_id=c."ид_канала_данных"
  LEFT JOIN state_pairs s ON c."тип_датчика"=s.sensor_type AND p.value_raw=s.value_raw
  GROUP BY ALL
)
SELECT * FROM by_type
""",
    "Покрытие справочника состояний",
)
display(
    state_coverage.groupby("source_year", as_index=False)[
        ["rows", "exact_state_text_matches"]
    ].sum()
)
print("Это совпадение текста и типа по текущему справочнику, а не проверенный физический исход.")

In [ ]:
systems = (
    year_groups.groupby("system_type", as_index=False)["rows"].sum().nlargest(10, "rows").iloc[::-1]
)
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(systems["system_type"], systems["rows"] / 1e6, color="#176a8a")
ax.set(
    title="События 2019–2026 по подсистемам текущего справочника",
    xlabel="Млн записей; текущая привязка каналов",
)
ax.grid(axis="x", alpha=0.25)
fig.tight_layout()
plt.show()

## 6. Единый обогащённый журнал: что именно соединяется

Одна исходная запись остаётся **одной строкой**. Обогащение начинается с `parquet_all`
(все восемь лет, уже сверенные с CSV по `3), а не с повторного чтения 313 млн CSV-строк.

| Источник | Связь с записью | Что добавляет | Граница достоверности |
| --- | --- | --- | --- |
| Годовые события | `channel_id`, дата и время | исходные `event_id`, `alarm`, `value_raw` | `event_id` не считаем глобально уникальным; текст и число сохраняем отдельно |
| Справочник каналов | `event.channel_id = channel.ид_канала_данных` | тип системы, тип сигнала, тег, название, ID объекта | **текущий снимок**, без дат действия; канал — сигнал/параметр, не обязательно отдельное физическое устройство |
| Справочник объектов | `channel.ид_объект = object.ид_объект` | уровень, родитель, вид, диспетчерское название | это иерархия для контекста/UI; историческая версия и физический смысл уровней неизвестны |
| Справочник состояний | `channel.тип_датчика + точный event.value_raw` | только **кандидатное** совпадение текста, число возможных наборов и конфликтов | у канала нет `ид_набор_состояний`; словарь не создаёт проверенную метку или подпись прогноза |
| Журнал-пример | прямой связи с историей нет | пример шестипольной схемы источника/API и UI | 1 августа 2026 года; не добавлять к годам и не считать обучающими исходами |

Перед соединением проверяем уникальность ID каналов и объектов. Справочник состояний
агрегируем до **одной строки на пару тип × точный текст**, чтобы он не размножал события.
`LEFT JOIN` сохраняет неизвестные каналы. Поля справочников обозначают **текущий
снимок**, а не доказанную историческую принадлежность.


In [ ]:
# Малые справочники: проверка ключей, неоднозначности и состава групп.
channel_key_duplicates = con.sql(
    'SELECT COUNT(*) - COUNT(DISTINCT "ид_канала_данных") FROM channels'
).fetchone()[0]
object_key_duplicates = con.sql(
    'SELECT COUNT(*) - COUNT(DISTINCT "ид_объект") FROM objects'
).fetchone()[0]
if channel_key_duplicates or object_key_duplicates:
    raise RuntimeError(
        f"Неуникальные ключи справочников: channels={channel_key_duplicates}, "
        f"objects={object_key_duplicates}"
    )

state_set_audit = con.sql(
    """
    SELECT "тип_датчика" AS sensor_type,
           "ид_набор_состояний" AS state_set_id,
           COUNT(*) AS state_rows,
           COUNT_IF("тревожное"='true') AS true_rows,
           COUNT_IF("тревожное"='false') AS false_rows
    FROM states
    GROUP BY ALL
    ORDER BY sensor_type, state_set_id
    """
).df()
print("Наборы состояний: число true не обязано равняться единице")
display(state_set_audit)

sensor_system_map = con.sql(
    """
    SELECT "тип_инж_системы" AS system_type,
           "тип_датчика" AS sensor_type,
           COUNT(*) AS current_channels,
           COUNT(DISTINCT "ид_объект") AS current_objects
    FROM channels
    GROUP BY ALL
    ORDER BY system_type, current_channels DESC
    """
).df()
print("Каналы текущего справочника: подсистема × тип сигнала")
display(sensor_system_map)
print(
    "Типов сигналов в нескольких подсистемах текущего снимка:",
    con.sql(
        """
        SELECT COUNT(*) FROM (
          SELECT "тип_датчика"
          FROM channels GROUP BY 1
          HAVING COUNT(DISTINCT "тип_инж_системы") > 1
        )
        """
    ).fetchone()[0],
)

object_structure = con.sql(
    """
    WITH RECURSIVE ancestry AS (
      SELECT "ид_объект" AS descendant_id,
             "ид_объект" AS ancestor_id,
             0 AS depth
      FROM objects
      UNION ALL
      SELECT a.descendant_id, parent."ид_объект" AS ancestor_id, a.depth+1
      FROM ancestry a
      JOIN objects child ON a.ancestor_id=child."ид_объект"
      JOIN objects parent ON child."родитель"=parent."ид_объект"
      WHERE a.depth<10
    ), direct AS (
      SELECT "ид_объект" AS object_id, COUNT(*) AS channel_count
      FROM channels GROUP BY 1
    ), by_object AS (
      SELECT o."ид_объект" AS object_id,
             o."иерархия_уровень" AS hierarchy_level,
             o."вид_объекта" AS object_kind,
             COALESCE(MAX(d0.channel_count),0) AS direct_channels,
             COALESCE(SUM(dd.channel_count),0) AS descendant_channels
      FROM objects o
      LEFT JOIN direct d0 ON o."ид_объект"=d0.object_id
      LEFT JOIN ancestry a ON o."ид_объект"=a.ancestor_id
      LEFT JOIN direct dd ON a.descendant_id=dd.object_id
      GROUP BY 1,2,3
    )
    SELECT hierarchy_level, object_kind,
           COUNT(*) AS objects,
           COUNT_IF(direct_channels>0) AS with_direct_channels,
           COUNT_IF(direct_channels=0 AND descendant_channels>0)
             AS with_only_descendant_channels,
           COUNT_IF(descendant_channels=0) AS without_known_descendant_channels
    FROM by_object
    GROUP BY ALL ORDER BY hierarchy_level, object_kind
    """
).df()
print("Объекты без прямого канала могут быть узлами с каналами у потомков")
display(object_structure)

In [ ]:
# Один тип сигнала принадлежит одному типу подсистемы в текущем справочнике.
from matplotlib.patches import Patch

sensor_chart = sensor_system_map.sort_values(
    ["system_type", "current_channels"], ascending=[True, False]
).reset_index(drop=True)
systems_in_chart = sorted(sensor_chart["system_type"].unique())
palette = plt.get_cmap("tab10")
system_colors = {system: palette(index) for index, system in enumerate(systems_in_chart)}
fig, ax = plt.subplots(figsize=(11, 8))
bars = ax.barh(
    sensor_chart["sensor_type"],
    sensor_chart["current_channels"],
    color=[system_colors[s] for s in sensor_chart["system_type"]],
)
ax.invert_yaxis()
ax.set(
    title="Текущий справочник: типы сигналов внутри подсистем",
    xlabel="Число каналов",
    ylabel="Тип сигнала",
)
ax.grid(axis="x", alpha=0.25)
ax.set_axisbelow(True)
ax.legend(
    handles=[Patch(facecolor=system_colors[s], label=s) for s in systems_in_chart],
    title="Тип подсистемы",
    loc="upper center",
    bbox_to_anchor=(0.5, -0.11),
    ncol=2,
    frameon=False,
)
fig.tight_layout()

### 6.1 Рабочий срез и схема обогащения

По выбранному для этого личного исследования правилу `enriched_df` исключает
**[2021-04-01, 2021-07-01)**: апрель, май и июнь 2021 года. Полный
`enriched_all_df` сохраняет их для диагностики. Это **исследовательская
чувствительность**, а не пересмотр действующей политики `exclude-2021-v1`:
для основной обучающей и оценочной выборки по-прежнему исключён весь 2021 год;
соответствующий ленивый срез ниже назван `enriched_policy_v1_df`. Окна
последовательностей нельзя склеивать через исключённый интервал.

`alarm_bool` — типизированный **исходный флаг той же записи**. `value_numeric` —
попытка прочитать число без потери `value_raw`; единицы и уставки зависят от типа
сигнала. `state_ref_*` — диагностические аннотации, **не признаки модели**.
Имена, теги и поля объекта могут быть кандидатами категориальных признаков
только после проверки версий справочника и времени доступности.


In [ ]:
# Одна строка на точную пару тип × текст; неоднозначность видна, не разрешается случайным выбором.
con.execute(
    """
    CREATE OR REPLACE TEMP TABLE state_ref_candidates AS
    SELECT "тип_датчика" AS sensor_type,
           "название_состояния" AS value_raw,
           COUNT(*) AS state_ref_rows,
           COUNT(DISTINCT "ид_набор_состояний") AS state_ref_set_count,
           COUNT(DISTINCT "тревожное") AS state_ref_alarm_count,
           CASE WHEN COUNT(DISTINCT "ид_набор_состояний")=1
                THEN MIN("ид_набор_состояний") END AS state_ref_candidate_set_id,
           CASE WHEN COUNT(DISTINCT "тревожное")=1
                THEN MIN("тревожное") END AS state_ref_alarm_consensus
    FROM states
    GROUP BY 1,2
    """
)
if con.sql(
    """
    SELECT COUNT(*)-COUNT(DISTINCT (sensor_type, value_raw))
    FROM state_ref_candidates
    """
).fetchone()[0]:
    raise RuntimeError("Повтор пары тип × текст после агрегации состояний")

# Ленивый слой: SQL-план, а не материализация сотен миллионов строк в pandas.
con.execute(
    """
    CREATE OR REPLACE VIEW enriched_all AS
    SELECT p.source_year, p.event_id, p.channel_id,
           p.event_date, p.event_time,
           TRY_CAST(p.event_date || ' ' || p.event_time AS TIMESTAMP) AS event_ts_local,
           p.alarm,
           CASE p.alarm WHEN 'true' THEN TRUE WHEN 'false' THEN FALSE END AS alarm_bool,
           p.value_raw,
           TRY_CAST(p.value_raw AS DOUBLE) AS value_numeric,
           c."тип_инж_системы" AS system_type,
           c."тип_датчика" AS sensor_type,
           c."тег_инженерной_системы" AS engineering_system_tag,
           c."название_датчика" AS sensor_name,
           c."ид_объект" AS object_id,
           TRY_CAST(o."иерархия_уровень" AS INTEGER) AS object_hierarchy_level,
           o."родитель" AS parent_object_id,
           o."вид_объекта" AS object_kind,
           o."диспетчерское_название_объекта" AS object_dispatch_name,
           c."ид_канала_данных" IS NOT NULL AS channel_ref_matched,
           o."ид_объект" IS NOT NULL AS object_ref_matched,
           s.sensor_type IS NOT NULL AS state_ref_exact_match,
           s.state_ref_rows,
           s.state_ref_set_count,
           s.state_ref_alarm_count,
           s.state_ref_candidate_set_id,
           s.state_ref_alarm_consensus,
           'current_snapshot_no_validity_dates' AS reference_basis
    FROM parquet_all p
    LEFT JOIN channels c ON p.channel_id=c."ид_канала_данных"
    LEFT JOIN objects o ON c."ид_объект"=o."ид_объект"
    LEFT JOIN state_ref_candidates s
      ON c."тип_датчика"=s.sensor_type AND p.value_raw=s.value_raw
    """
)
con.execute(
    """
    CREATE OR REPLACE VIEW enriched_working AS
    SELECT * FROM enriched_all
    WHERE NOT (
      source_year=2021
      AND event_date>='2021-04-01' AND event_date<'2021-07-01'
    )
    """
)
con.execute(
    "CREATE OR REPLACE VIEW enriched_policy_v1 AS "
    "SELECT * FROM enriched_all WHERE source_year<>2021"
)
enriched_all_df = con.sql("SELECT * FROM enriched_all")
enriched_df = con.sql("SELECT * FROM enriched_working")
enriched_policy_v1_df = con.sql("SELECT * FROM enriched_policy_v1")

print("enriched_df — один ленивый датафрейм; типы DuckDB:")
display(con.sql("DESCRIBE SELECT * FROM enriched_working").df())
print("Пример соединённой строки (только ограниченный просмотр):")
display(
    con.sql(
        """
        SELECT source_year, event_id, channel_id, event_date, event_time,
               alarm, value_raw, value_numeric, system_type, sensor_type,
               sensor_name, object_id, object_kind, state_ref_exact_match
        FROM enriched_working WHERE source_year=2019 LIMIT 5
        """
    ).df()
)
print("Не вызывайте enriched_df.df() без LIMIT: полный слой превышает объём RAM.")

### 6.2 Проверка сохранения строк и покрытия

Следующий проход читает рабочий срез целиком, но возвращает только агрегат
`год × объект текущего справочника`. Таймер из `2 показывает ход долгого SQL.
Для каждого года итог после всех `LEFT JOIN` должен равняться исходному числу
строк; для 2021 вычитаем только три выбранных месяца. Неизвестные каналы
сохраняются с пустыми полями справочников. Счётчики совпадений состояний
не делают словарь полным или исторически верным.


In [ ]:
excluded_2021_q2 = con.sql(
    """
    SELECT COUNT(*) FROM parquet_all
    WHERE source_year=2021
      AND event_date>='2021-04-01' AND event_date<'2021-07-01'
    """
).fetchone()[0]
expected_by_year = {int(row["source_year"]): int(row["rows"]) for row in parquet_stats}
expected_by_year[2021] -= excluded_2021_q2

object_year_audit = fetch_small(
    """
    SELECT source_year, object_id, channel_ref_matched, object_ref_matched,
           COUNT(*) AS event_rows,
           COUNT(DISTINCT channel_id) AS channels_with_messages,
           COUNT_IF(state_ref_exact_match) AS state_text_matches,
           COUNT_IF(alarm_bool) AS alarm_true
    FROM enriched_working
    GROUP BY ALL
    ORDER BY source_year, object_id
    """,
    "Обогащение: учёт строк и покрытия по всем годам",
)
year_audit = object_year_audit.groupby("source_year", as_index=False)[
    ["event_rows", "channels_with_messages", "state_text_matches", "alarm_true"]
].sum()
year_audit["expected_rows"] = year_audit["source_year"].map(expected_by_year)
year_audit["current_objects_with_messages"] = year_audit["source_year"].map(
    object_year_audit[object_year_audit["object_ref_matched"]]
    .groupby("source_year")["object_id"]
    .nunique()
)
year_audit["unmatched_channel_rows"] = (
    year_audit["source_year"]
    .map(
        object_year_audit[~object_year_audit["channel_ref_matched"]]
        .groupby("source_year")["event_rows"]
        .sum()
    )
    .fillna(0)
    .astype("int64")
)
year_audit["unmatched_object_rows"] = (
    year_audit["source_year"]
    .map(
        object_year_audit[
            object_year_audit["channel_ref_matched"] & ~object_year_audit["object_ref_matched"]
        ]
        .groupby("source_year")["event_rows"]
        .sum()
    )
    .fillna(0)
    .astype("int64")
)
if not year_audit["event_rows"].eq(year_audit["expected_rows"]).all():
    raise AssertionError("Соединение изменило число записей; проверьте ключи и SQL")
if len(year_audit) != len(YEARS):
    raise AssertionError("После фильтра неожиданно пропал год")
print(f"Исключено записей апреля–июня 2021: {excluded_2021_q2:,}")
display(
    year_audit[
        [
            "source_year",
            "expected_rows",
            "event_rows",
            "unmatched_channel_rows",
            "unmatched_object_rows",
            "current_objects_with_messages",
            "state_text_matches",
            "alarm_true",
        ]
    ]
)

### 6.3 Что этот слой даёт для признаков и UI

- `channel_id` — ключ истории **сигнала/параметра**. В текущем справочнике каждому
  каналу назначены один объект, один тип инженерной системы и один тип сигнала.
  Административная группа `object_id × system_type` не доказана как один физический агрегат.
- Название и тег сохраняются как строки. Суффиксы вроде «Дым ПК826+4» могут
  кодировать место, но пока нет подтверждённого словаря. Смешивать цифры из
  имени с `value_numeric` как физическую величину или уставку нельзя.
- Объекты без **прямых** каналов могут быть родительскими узлами с каналами у
  потомков. Для UI различать прямые каналы, каналы потомков и фактически
  наблюдавшиеся сообщения. Нулевое число сообщений не означает норму или
  отсутствие риска.
- Текст исходного события, исходный `alarm` и будущий прогноз показывать
  отдельно. Справочник состояний не позволяет по ответу модели «0/1» выбрать
  конкретную подпись: у типа бывает несколько наборов, наборы имеют разное
  число `true`, а связи набора с каналом нет.
- Для прогноза на следующие сутки единица могла бы быть `канал × момент t`:
  признаки только из сообщений **до t**, будущая регистрация alarm или начало
  серии — отдельная метка; при недостаточной наблюдаемости исход неизвестен.
  Текущее `value_raw` может быть признаком только если запись уже поступила
  до t. Одна модель для всех систем и отдельные модели по типам — варианты
  для последующего честного сравнения, не вывод из этого join.

**Следующий опыт о связях:** сравнить совместные начала и лаги 0/1/6/24 ч
между каналами (а) одного объекта и типа системы, (б) того же объекта, но
разных систем, (в) сопоставимых других объектов. Знаменатель — интервалы,
когда обе стороны наблюдаемы; отдельно контролировать частоту сообщений,
пакетные всплески, повторы и календарь. Устойчивая совместность поддержит
идею контекстных признаков, но не докажет общую физическую причину.

Этот блокнот строит описательный слой. Он не утверждает новую метку, допуск
обучения или решения D01/D05/D06; для действующей модели остаётся
`exclude-2021-v1` и отдельная постановка газового F1.
